# Baum-Welch Algorithm for Hidden Markov Models

This notebook implements the Baum-Welch algorithm for parameter estimation in Hidden Markov Models (HMMs). The Baum-Welch algorithm is an Expectation-Maximization (EM) approach that learns HMM parameters from unlabeled observation sequences by iteratively refining:

- **Initial state probabilities** (betas)
- **Transition probabilities**
- **Emission probabilities**

### Algorithm Steps
1. **Initialization** — Set initial guesses for all parameters
2. **E-Step** — Run Forward-Backward; compute γ (gamma) and ξ (xi) expected counts
3. **M-Step** — Re-estimate parameters from expected counts
4. **Convergence** — Repeat until log-likelihood change < tolerance

## 1. Imports

In [1]:
import numpy as np
from numbers import Number
from collections.abc import Iterable
from math import log

## 2. State Class

In [2]:
class State:
    """Hidden state for HMM"""
    def __init__(self, name: str, emissions: list, probabilities: list[Number], transitions: dict[str:float]):
        self.name = name
        self.emissions = set(emissions)
        self.emission_probs = dict(zip(emissions, probabilities))
        self.transitions = transitions
        self.total_emission_prob = sum(probabilities)
        if not np.isclose(self.total_emission_prob, 1.0):
            raise ValueError("Emission probabilities do not sum to 1")

    def __repr__(self):
        return self.name

    def add_emission(self, emission, probability: Number):
        self.emissions.add(emission)
        self.emission_probs[emission] = probability
        self.total_emission_prob += probability
        if self.total_emission_prob > 1:
            print(f"\nWARNING: sum of State: {self.name} emission probabilities has exceeded 1.\n"
                  f"Refactoring to maintain relative probabilities with sum of 1\n")
            for emit in self.emissions:
                self.emission_probs[emit] = self.emission_probs[emit] / self.total_emission_prob
            self.total_emission_prob = sum(self.emission_probs.values())

## 3. HMM Class

All methods are defined inside the class:
- **Forward & Backward** — from last week's project; foundation for the E-step
- **Posterior Decoding** — most likely state at each position using γ
- **E-step helpers** — `_expectation_values`, `gamma_matrix`, `xi_tensor`
- **Baum-Welch** — full EM loop

In [3]:
class HMM():
    def __init__(self, name: str, betas: dict[State:float], emissions: set, states: list[State]):
        self.name = name
        self.states = states
        self.t_mat = self.build_transition_mat_from_states()
        self.emissions = set(emissions)
        self.betas = betas
        self._synchronize_emission()

    def __repr__(self):
        return (f"{self.name}\n{self.emissions}\n{self.states}\n{self.t_mat}\n{self.betas}")

    @staticmethod
    def _safe_log(x: float) -> float:
        """Return log(x) if x > 0, else -inf"""
        if x <= 0:
            return float("-inf")
        return log(x)

    def _synchronize_emission(self):
        """Ensure every state has every emission in the HMM. Missing emissions get probability 0."""
        for state in self.states:
            for emission in state.emissions:
                self.emissions.add(emission)
        for emission in self.emissions:
            for state in self.states:
                if emission not in state.emissions:
                    state.emissions.add(emission)
                    state.emission_probs[emission] = 0.0

    def build_transition_mat_from_states(self):
        """Build a transition matrix from states' transition dicts."""
        t_mat = [[] for state in self.states]
        for row, state in enumerate(self.states):
            for trans in self.states:
                t_mat[row].append(state.transitions[trans.name])
        return np.array(t_mat, dtype=float)

    def add_state(self, name: str = None, emissions: list = None, probabilities: list = None,
                  transitions: dict[str, float] = None, state: State = None):
        if emissions is None:
            emissions = []
        if probabilities is None:
            probabilities = []
        if state is None:
            if name is None:
                raise ValueError("No name given to create new State object from argument")
            if transitions is None:
                raise ValueError("No transitions given to create new State object from argument")
            state = State(name=name, emissions=emissions, probabilities=probabilities, transitions=transitions)
        self.states.append(state)
        self._synchronize_emission()
        self.t_mat = self.build_transition_mat_from_states()

    def _get_prev_state_options(self, obs: int, observations: Iterable, mat: np.ndarray, mat_row: int) -> list:
        options = [
            mat[trans_state][obs - 1]
            + self._safe_log(self.t_mat[trans_state][mat_row])
            + self._safe_log(self.states[mat_row].emission_probs[observations[obs]])
            for trans_state in range(len(self.states))
        ]
        return options

    def _get_future_options(self, obs_ind: int, observations: Iterable, mat: np.ndarray, mat_row: int) -> list:
        options = [
            self._safe_log(self.t_mat[mat_row][next_state])
            + self._safe_log(self.states[next_state].emission_probs[observations[obs_ind]])
            + mat[next_state][obs_ind]
            for next_state in range(len(self.states))
        ]
        return options

    # ------------------------------------------------------------------
    # Forward algorithm (from last week)
    # ------------------------------------------------------------------

    def forward(self, observations: Iterable) -> np.ndarray:
        """
        Create forward matrix from sequence of observations.

        Args:
            observations (Iterable): sequence of observations

        Returns:
            np.ndarray: matrix of cumulative path probabilities for each state at each position
        """
        forward_mat = np.full((len(self.states), len(observations)), float("-inf"))
        for state, row in enumerate(forward_mat):
            beta = self.betas[self.states[state]]
            emission = self.states[state].emission_probs[observations[0]]
            row[0] = self._safe_log(beta) + self._safe_log(emission)
        for obs_ind in range(1, len(observations)):
            for state, row in enumerate(forward_mat):
                options = self._get_prev_state_options(obs=obs_ind, observations=observations,
                                                       mat=forward_mat, mat_row=state)
                row[obs_ind] = np.logaddexp.reduce(options)
        return forward_mat

    # ------------------------------------------------------------------
    # Backward algorithm (from last week)
    # ------------------------------------------------------------------

    def backward(self, observations: Iterable) -> np.ndarray:
        """
        Create backward matrix from sequence of observations.

        Args:
            observations (Iterable): sequence of observations

        Returns:
            np.ndarray: backward matrix
        """
        backward_mat = np.full((len(self.states), len(observations)), float("-inf"))
        for row in backward_mat:
            row[-1] = 0.0
        for obs_ind in range(len(observations) - 1, 0, -1):
            for state, row in enumerate(backward_mat):
                options = self._get_future_options(observations=observations, obs_ind=obs_ind,
                                                   mat=backward_mat, mat_row=state)
                row[obs_ind - 1] = np.logaddexp.reduce(options)
        return backward_mat

    # ------------------------------------------------------------------
    # Posterior decoding (from last week)
    # ------------------------------------------------------------------

    def posterier_decoding(self, observations: Iterable):
        """
        Calculate the most likely state at each position using the forward-backward algorithm.

        Args:
            observations (Iterable): sequence of observations

        Returns:
            (np.ndarray, list): posterior decoding matrix and list of most likely state at each position
        """
        forward_mat = self.forward(observations)
        backward_mat = self.backward(observations)
        log_prob = np.logaddexp.reduce(forward_mat[:, -1])
        posterior_mat = forward_mat + backward_mat - log_prob
        state_idx = np.argmax(posterior_mat, axis=0)
        state_path = [self.states[i] for i in state_idx]
        return posterior_mat, state_path

    # ------------------------------------------------------------------
    # E-step helpers
    # ------------------------------------------------------------------

    def _expectation_values(self, observations: Iterable):
        """Compute shared E-step quantities for one sequence."""
        observations = list(observations)
        n_states = len(self.states)
        T = len(observations)

        forward_mat = self.forward(observations)
        backward_mat = self.backward(observations)
        log_prob = np.logaddexp.reduce(forward_mat[:, -1])

        gamma = np.exp(forward_mat + backward_mat - log_prob)

        xi = np.zeros((T - 1, n_states, n_states), dtype=float)
        for t in range(T - 1):
            for i in range(n_states):
                for j in range(n_states):
                    log_xi = (
                        forward_mat[i, t]
                        + self._safe_log(self.t_mat[i, j])
                        + self._safe_log(self.states[j].emission_probs[observations[t + 1]])
                        + backward_mat[j, t + 1]
                        - log_prob
                    )
                    xi[t, i, j] = np.exp(log_xi)

        return forward_mat, backward_mat, log_prob, gamma, xi

    def sequence_log_likelihood(self, observations: Iterable) -> float:
        """Compute log-likelihood of a sequence of observations."""
        _, _, log_prob, _, _ = self._expectation_values(observations)
        return log_prob

    def gamma_matrix(self, observations: Iterable) -> np.ndarray:
        """Return gamma matrix."""
        _, _, _, gamma, _ = self._expectation_values(observations)
        return gamma

    def xi_tensor(self, observations: Iterable) -> np.ndarray:
        """Return xi tensor."""
        _, _, _, _, xi = self._expectation_values(observations)
        return xi

    def posterior_decoding(self, observations: Iterable):
        """Compute posterior decoding using gamma matrix."""
        gamma = self.gamma_matrix(observations)
        posterior_mat = np.where(gamma > 0, np.log(gamma), float("-inf"))
        state_idx = np.argmax(posterior_mat, axis=0)
        state_path = [self.states[i] for i in state_idx]
        return posterior_mat, state_path

    # ------------------------------------------------------------------
    # Baum-Welch
    # ------------------------------------------------------------------

    def baum_welch(self, sequences, max_iter=100, tol=1e-4, pseudocount=1e-6):
        """
        Estimate HMM parameters from training sequences using the Baum-Welch (EM) algorithm.

        Args:
            sequences   (list): list of observation sequences
            max_iter    (int):  maximum number of EM iterations
            tol         (float): convergence threshold on log-likelihood change
            pseudocount (float): added to all counts to prevent zero probabilities

        Returns:
            history (list): total log-likelihood after each iteration
        """
        n_states = len(self.states)
        emission_symbols = sorted(self.emissions)
        emission_to_idx = {}
        for i, symbol in enumerate(emission_symbols):
            emission_to_idx[symbol] = i

        history = []
        prev_loglik = float("-inf")

        for iteration in range(max_iter):

            beta_counts  = np.zeros(n_states)
            trans_counts = np.zeros((n_states, n_states))
            trans_denom  = np.zeros(n_states)
            emit_counts  = np.zeros((n_states, len(emission_symbols)))
            emit_denom   = np.zeros(n_states)
            total_loglik = 0.0

            # -------- E-STEP --------
            for observations in sequences:
                observations = list(observations)
                _, _, log_prob, gamma, xi = self._expectation_values(observations)

                total_loglik += log_prob
                beta_counts  += gamma[:, 0]
                trans_counts += np.sum(xi, axis=0)
                trans_denom  += np.sum(gamma[:, :-1], axis=1)

                for t in range(len(observations)):
                    obs = observations[t]
                    obs_idx = emission_to_idx[obs]
                    emit_counts[:, obs_idx] += gamma[:, t]

                emit_denom += np.sum(gamma, axis=1)

            # -------- M-STEP --------

            # Update initial probabilities
            beta_total = np.sum(beta_counts) + pseudocount * n_states
            for i in range(n_states):
                self.betas[self.states[i]] = (beta_counts[i] + pseudocount) / beta_total

            # Update transition probabilities
            for i in range(n_states):
                denom = trans_denom[i] + pseudocount * n_states
                for j in range(n_states):
                    prob = (trans_counts[i, j] + pseudocount) / denom
                    self.t_mat[i, j] = prob
                    self.states[i].transitions[self.states[j].name] = prob

            # Update emission probabilities
            for i in range(n_states):
                denom = emit_denom[i] + pseudocount * len(emission_symbols)
                for symbol in emission_symbols:
                    k = emission_to_idx[symbol]
                    prob = (emit_counts[i, k] + pseudocount) / denom
                    self.states[i].emission_probs[symbol] = prob

            history.append(total_loglik)

            # Convergence check
            if len(history) > 1 and abs(total_loglik - prev_loglik) < tol:
                break

            prev_loglik = total_loglik

        return history

## 4. Running the Algorithm

### 4.1 Toy Example — Verify Forward & Backward

A simple 2-state model with alphabet `{A, B, C}` to confirm the forward and backward matrices and posterior decoding work correctly.

In [4]:
observations = "ABACB"

my_state = State(name="my_state", emissions=["A", "B", "C"], probabilities=[0.3, 0.2, 0.5],
                 transitions={"my_state": 0.7, "my_state2": 0.3})
my_state2 = State(name="my_state2", emissions=["A", "B", "C"], probabilities=[0.2, 0.7, 0.1],
                  transitions={"my_state2": 0.9, "my_state": 0.1})
my_HMM = HMM(name="My_HMM", betas={my_state: 0.5, my_state2: 0.5},
             emissions={"A", "B", "C"}, states=[my_state, my_state2])

fwmat = my_HMM.forward(observations=observations)
bkmat = my_HMM.backward(observations=observations)
print(fwmat)
print("np.logaddexp(fwmat[0][-1], fwmat[1][-1])", np.logaddexp(fwmat[0][-1], fwmat[1][-1]))
print(bkmat)
print("np.logaddexp(bkmat[0][-1], bkmat[1][-1])", np.logaddexp(bkmat[0][0], bkmat[1][0]))

post_mat, post_path = my_HMM.posterier_decoding(observations=observations)
print(post_mat)
print(post_path)

log_prob_fw = np.logaddexp.reduce(fwmat[:, -1])
log_prob_bk = np.logaddexp.reduce(bkmat[:, 0])
print("log_P from forward: ", log_prob_fw)
print("log_P from backward:", log_prob_bk)

[[-1.89711998 -3.77226106 -4.87109077 -5.62619663 -7.52021504]
 [-2.30258509 -2.35915544 -3.99594824 -6.27380093 -6.2429798 ]]
np.logaddexp(fwmat[0][-1], fwmat[1][-1]) -5.997052128607013
[[-4.75825144 -3.37028028 -1.95192822 -1.04982212  0.        ]
 [-4.42369899 -4.02072242 -2.57702194 -0.43078292  0.        ]]
np.logaddexp(bkmat[0][-1], bkmat[1][-1]) -3.8839021297114904
[[-0.65831929 -1.14548921 -0.82596686 -0.67896662 -1.52316291]
 [-0.72923196 -0.38282574 -0.57591805 -0.70753172 -0.24592767]]
[my_state, my_state2, my_state2, my_state, my_state2]
log_P from forward:  -5.997052128607013
log_P from backward: -3.8839021297114904


### 4.2 Baum-Welch — GC Content HMM

A 2-state HMM for DNA sequences with states **H** (high GC) and **L** (low GC). We train on three short DNA sequences and observe how parameters update after running Baum-Welch.

In [5]:
obs = ["GGCACTGAA", "ATGCAATGC", "AATGCCTGA"]
sequences = [list(seq) for seq in obs]

state_H = State(
    name="H",
    emissions=["A", "C", "G", "T"],
    probabilities=[0.2, 0.3, 0.3, 0.2],
    transitions={"H": 0.6, "L": 0.4}
)
state_L = State(
    name="L",
    emissions=["A", "C", "G", "T"],
    probabilities=[0.3, 0.2, 0.2, 0.3],
    transitions={"H": 0.3, "L": 0.7}
)
my_HMM = HMM(
    name="GC_Model",
    betas={state_H: 0.5, state_L: 0.5},
    emissions={"A", "C", "G", "T"},
    states=[state_H, state_L]
)

print("--- Initial Parameters ---")
print("Initial betas:")
for state in my_HMM.states:
    print(state.name, my_HMM.betas[state])

print("\nInitial transition matrix:")
print(my_HMM.t_mat)

print("\nInitial emission probabilities:")
for state in my_HMM.states:
    print(state.name, state.emission_probs)

--- Initial Parameters ---
Initial betas:
H 0.5
L 0.5

Initial transition matrix:
[[0.6 0.4]
 [0.3 0.7]]

Initial emission probabilities:
H {'A': 0.2, 'C': 0.3, 'G': 0.3, 'T': 0.2}
L {'A': 0.3, 'C': 0.2, 'G': 0.2, 'T': 0.3}


In [12]:
history = my_HMM.baum_welch(sequences, max_iter=20, tol=1e-4, pseudocount=1e-6)

print("--- Baum-Welch Log-Likelihood History ---")
print(history)

print("\n--- Updated Parameters After Baum-Welch ---")
print("Updated betas:")
for state in my_HMM.states:
    print(state.name, my_HMM.betas[state])

print(sum(my_HMM.betas[state] for state in my_HMM.states))

print("\nUpdated transition matrix:")
print(my_HMM.t_mat)

for i, state in enumerate(my_HMM.states):
    print(f"{state.name} row sum =", np.sum(my_HMM.t_mat[i]))

print("\nUpdated emission probabilities:")
for state in my_HMM.states:
    print(state.name, state.emission_probs)

print(sum(my_HMM.states[0].emission_probs.values()))
print(sum(my_HMM.states[1].emission_probs.values()))

--- Baum-Welch Log-Likelihood History ---
[np.float64(-33.986454716076125), np.float64(-33.98643193335699)]

--- Updated Parameters After Baum-Welch ---
Updated betas:
H 4.6485885205113486e-07
L 0.999999535141148
1.0

Updated transition matrix:
[[0.36366547 0.63633453]
 [0.74258255 0.25741745]]
H row sum = 1.0
L row sum = 0.9999999999999994

Updated emission probabilities:
H {'A': np.float64(0.18074771525076067), 'C': np.float64(0.44686292710975467), 'G': np.float64(2.108106832048869e-06), 'T': np.float64(0.3723872495326527)}
L {'A': np.float64(0.4842748992395226), 'C': np.float64(1.879810381183016e-06), 'G': np.float64(0.515723090911295), 'T': np.float64(1.3003880119870542e-07)}
1.0
0.9999999999999999
